# Clasificación de 3 escenas con DenseNet121

DenseNet es la arquitectura **no secuencial** que pide el enunciado: dentro de
un bloque denso, la entrada de la capa `l` es la **concatenación** de las
salidas de las capas `0..l-1`. Gracias a esa conectividad densa cada capa
reutiliza las características ya extraídas por todas las anteriores, el número
de parámetros se reduce (ninguna capa tiene que reaprender de cero) y el
gradiente fluye mejor hacia las primeras capas.

Se parte de `DenseNet121` **preentrenada en ImageNet** y se adapta la cabeza a
las tres clases del reto (`coast`, `forest`, `highway`) en dos fases:

1. **Fase 1** — base congelada: solo se entrena la cabeza clasificadora con
   `Adam(1e-3)`. Congelar la base evita que los gradientes grandes de una
   cabeza inicializada al azar destruyan los filtros genéricos aprendidos en
   ImageNet (bordes, texturas, formas), y hace la fase rápida y estable.
2. **Fase 2** — fine-tuning: se descongelan las últimas 40 capas (final del
   bloque `conv5`), **excepto las `BatchNormalization`**, y se reentrena con un
   learning rate bajo (`Adam(1e-5)`). Las capas `BatchNormalization` se quedan
   congeladas porque sus medias y varianzas acumuladas se aprendieron sobre
   ImageNet: reajustarlas con ~720 imágenes de train y batches pequeños
   desestabiliza el modelo.

La primera ejecución descarga los pesos de ImageNet (`weights="imagenet"`).
Las imágenes se redimensionan a 128×128 y se preprocesan con
`keras.applications.densenet.preprocess_input` **dentro del propio modelo**.
El modelo se carga desde `resultados/densenet/` por defecto; cambia
`FORCE_RETRAIN = True` para reentrenarlo.

## Configuración

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from keras import layers
from keras.applications import DenseNet121
from keras.applications.densenet import preprocess_input
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam

import common_3scenes as cs

FORCE_RETRAIN = False
sns.set_theme(style="whitegrid", context="notebook")
keras.utils.set_random_seed(cs.SEED)
np.random.seed(cs.SEED)

## Carga del dataset y split común

In [ ]:
data = cs.load_split_data()
X_train = data["X_train"].astype("float32")
X_val = data["X_val"].astype("float32")
X_test = data["X_test"].astype("float32")
y_train_labels = data["y_train"]
y_val_labels = data["y_val"]
y_test_labels = data["y_test"]
test_paths = data["test_paths"]

y_train = cs.one_hot(y_train_labels)
y_val = cs.one_hot(y_val_labels)

print("Train:", X_train.shape, "| Validación:", X_val.shape, "| Test:", X_test.shape)
print("Distribución de test:", {name: int(np.sum(y_test_labels == name)) for name in cs.CLASSES})

## Modelo, entrenamiento y evaluación

El modelo es funcional (no `Sequential`): `Input(128, 128, 3)` → augmentación
(`RandomFlip("horizontal")`, `RandomRotation(0.08)`, `RandomZoom(0.10)`,
`RandomTranslation(0.08, 0.08)`; solo activa en entrenamiento) →
`layers.Lambda(preprocess_input)` → base `DenseNet121(include_top=False,
weights="imagenet", input_shape=(128, 128, 3))` con `base.trainable = False` →
`GlobalAveragePooling2D()` → `Dropout(0.35)` → `Dense(3, softmax)`.

Las dos fases usan batch de 32 y 12 épocas como máximo, con `EarlyStopping`
sobre `val_accuracy` (con `restore_best_weights`) y `ReduceLROnPlateau` sobre
`val_loss`. En la fase 2 se recompila con `Adam(1e-5)` después de descongelar
las últimas 40 capas saltando las `BatchNormalization`. Las dos historias se
concatenan en un único `history.csv` con índice de época 1-based.

In [ ]:
def build_densenet():
    inputs = layers.Input(shape=(128, 128, 3))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.08)(x)
    x = layers.RandomZoom(0.10)(x)
    x = layers.RandomTranslation(0.08, 0.08)(x)
    x = layers.Lambda(preprocess_input, name="densenet_preprocessing")(x)

    base = DenseNet121(include_top=False, weights="imagenet", input_shape=(128, 128, 3))
    base.trainable = False

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    outputs = layers.Dense(len(cs.CLASSES), activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="densenet121_3scenes")
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=1e-3),
        metrics=["accuracy"],
    )
    return model


def fit_densenet(model):
    base = model.get_layer("densenet121")
    phase1_callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ]
    history_phase1, seconds_phase1 = cs.train_keras_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=12,
        batch_size=32,
        callbacks=phase1_callbacks,
    )

    for layer in base.layers[-40:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True
    model.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=1e-5),
        metrics=["accuracy"],
    )

    phase2_callbacks = [
        EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ]
    history_phase2, seconds_phase2 = cs.train_keras_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        epochs=12,
        batch_size=32,
        callbacks=phase2_callbacks,
    )

    history = pd.concat(
        [pd.DataFrame(history_phase1.history), pd.DataFrame(history_phase2.history)],
        ignore_index=True,
    )
    history.index = np.arange(1, len(history) + 1)
    return model, history, seconds_phase1 + seconds_phase2


def evaluate_densenet(model, history, training_seconds):
    predicted_idx, probabilities, inference_ms = cs.predict_keras(model, X_test)
    predicted = np.array(cs.CLASSES)[predicted_idx]
    metrics = cs.compute_metrics(
        name="DenseNet121",
        family="DL (non-sequential)",
        y_true=y_test_labels,
        y_pred=predicted,
        y_prob=probabilities,
        params=model.count_params(),
        trainable_params=sum(np.prod(weight.shape) for weight in model.trainable_weights),
        training_seconds=training_seconds,
        inference_ms=inference_ms,
        epochs=len(history),
        hyperparameters={
            "base": "DenseNet121 ImageNet",
            "input": 128,
            "phase1_lr": 1e-3,
            "phase2_lr": 1e-5,
            "batch_size": 32,
            "epochs_phase1": 12,
            "epochs_phase2": 12,
            "dropout": 0.35,
            "fine_tuning_layers": 40,
        },
        notes="DenseNet121 pretrained on ImageNet with two-phase fine-tuning (BatchNorm layers stay frozen).",
    )
    cs.save_results(
        "densenet",
        metrics,
        y_test_labels,
        predicted,
        probabilities,
        test_paths=test_paths,
        history=history,
        model=model,
    )
    return metrics

In [ ]:
model, history, metrics, retrained = cs.load_or_train(
    "densenet",
    build_model=build_densenet,
    fit_model=fit_densenet,
    evaluate_model=evaluate_densenet,
    force=FORCE_RETRAIN,
)
print("Accuracy en test:", round(metrics["accuracy"], 4))

## Grad-CAM: ¿dónde mira la red?

Grad-CAM toma los gradientes de la puntuación de cada clase respecto a la
última capa convolucional (`conv5_block16_concat`) y pondera sus canales para
pintar un mapa de calor: las zonas que más empujan la decisión de esa clase.
Se elige una imagen de test (aleatoria con la semilla del reto) y se muestra su
mapa de calor **para cada una de las tres clases**, una por columna (arriba,
superpuesto a la imagen; abajo, el mapa de calor solo).

La celda vive **después** de la celda de cargar-o-entrenar y está protegida con
`try/except`: si algo falla (por ejemplo, el modelo en caché no expone la capa
esperada), se imprime un aviso y el resto del cuaderno sigue sin romperse. La
figura se guarda como evidencia en `resultados/densenet/gradcam.png`.

In [ ]:
try:
    base = model.get_layer("densenet121")
    conv_layer = base.get_layer("conv5_block16_concat")
    grad_model = keras.Model(base.input, [conv_layer.output, base.output])

    gap = next(layer for layer in model.layers if isinstance(layer, layers.GlobalAveragePooling2D))
    dropout = next(layer for layer in model.layers if isinstance(layer, layers.Dropout))
    classifier = next(layer for layer in model.layers if isinstance(layer, layers.Dense))

    def grad_cam(image, class_index):
        with tf.GradientTape() as tape:
            conv_output, features = grad_model(
                preprocess_input(image[None].astype("float32")), training=False
            )
            predictions = classifier(dropout(gap(features), training=False))
            loss = predictions[:, class_index]
        gradients = tape.gradient(loss, conv_output)
        weights = tf.reduce_mean(gradients, axis=(0, 1, 2))
        heatmap = tf.reduce_mean(conv_output[0] * weights, axis=-1)
        heatmap = tf.maximum(heatmap, 0)
        heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
        return heatmap.numpy()

    rng = np.random.default_rng(cs.SEED)
    index = int(rng.integers(len(X_test)))
    image = X_test[index]
    probabilities = model.predict(image[None], verbose=0)[0]

    fig, axes = plt.subplots(2, len(cs.CLASSES), figsize=(4.6 * len(cs.CLASSES), 7.4))
    for class_index, class_name in enumerate(cs.CLASSES):
        heatmap = grad_cam(image, class_index)
        heatmap = tf.image.resize(heatmap[..., None], (128, 128)).numpy()[..., 0]

        axes[0, class_index].imshow(image.astype("uint8"))
        axes[0, class_index].imshow(heatmap, cmap="jet", alpha=0.45)
        axes[0, class_index].set_title(
            f"Clase: {class_name}\nP({class_name}) = {probabilities[class_index]:.3f}",
            fontsize=10,
        )
        axes[0, class_index].axis("off")

        axes[1, class_index].imshow(heatmap, cmap="jet")
        axes[1, class_index].set_title(f"Mapa de calor ({class_name})", fontsize=10)
        axes[1, class_index].axis("off")

    predicted_index = int(np.argmax(probabilities))
    fig.suptitle(
        f"Grad-CAM sobre DenseNet121 — imagen de test: {y_test_labels[index]}, "
        f"predicción: {cs.CLASSES[predicted_index]}",
        y=1.02,
    )
    fig.tight_layout()
    cs.result_dir("densenet").mkdir(parents=True, exist_ok=True)
    fig.savefig(cs.result_dir("densenet") / "gradcam.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as exc:
    print(f"[aviso] No se pudo generar el Grad-CAM: {exc}")

## Observaciones

* El preentrenamiento en ImageNet aporta filtros genéricos de bordes,
  texturas y formas; el fine-tuning en dos fases los especializa en costa,
  bosque y autopista con muy pocas épocas frente a entrenar desde cero.
* La conectividad densa reutiliza características de todas las capas
  anteriores: menos parámetros entrenables por bloque y un gradiente más sano
  que en una CNN plana.
* El Grad-CAM deja evidencia de interpretabilidad: se espera que la red mire
  al agua y a la línea de costa en `coast`, a la masa de vegetación en
  `forest` y a las calzadas, líneas y bordillos en `highway`.
* Los artefactos completos quedan en `resultados/densenet/` (`metrics.json`,
  `predictions.csv`, `classification_report.txt`, `history.csv` y
  `model.keras`) y la figura en `resultados/densenet/gradcam.png`. La
  comparación con el resto de modelos, en `reto-1.ipynb`.